## Self notes — running log

Short observations only. One markdown cell per note. Add a code cell underneath if a demo is genuinely useful — most of the time it isn't, the practice notebooks already cover that.


### 1 — `SparkSession` has two versions

- `pyspark.sql.session.SparkSession` — **classic**, drives a local JVM.
- `pyspark.sql.connect.session.SparkSession` — **Connect**, talks to a remote Spark server over gRPC.

Same API surface, different classes. Helpers that need to touch the JVM directly (e.g. `delta.configure_spark_with_delta_pip(builder)`) accept only the classic builder and reject the Connect one with a type-mismatch error.


### 2 — Predicate pushdown: what does and doesn't push

- **In-memory sources can never push filters.** `Range` (`spark.range`), `LocalTableScan` / `ExistingRDD` (`createDataFrame`) have no scan layer to push into — the data is already in memory. The `Filter` just sits above the source.
- **Even when a filter pushes, the `Filter` node stays in the plan.** Source-level pushdown is best-effort (e.g. Parquet uses row-group min/max stats — it can skip whole groups but rows inside surviving groups still need exact re-checking). Catalyst keeps the `Filter` above the scan for correctness.

```
*(1) Project [customer_id#0, full_name#1, credit_score#3L]
  +- *(1) Filter (isnotnull(credit_score#3L) AND (credit_score#3L >= 700))   ← exact re-check
     +- *(1) ColumnarToRow
        +- FileScan parquet [...]
             PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)]
             ReadSchema: struct<customer_id:string,full_name:string,credit_score:bigint>
```


---

_New entries go below. Template:_

```
### N — title

One- or two-sentence observation. Code only if it adds something the practice notebooks don't already show.
```
